In [1]:
# Import libries
import numpy as np
import pandas as pd
import os
import re
import glob
import json
import csv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyranges as pr
import pysam
# from scipy.stats import mannwhitneyu, stats
# from statannotations.Annotator import Annotator

current_directory = os.getcwd()
print("Current Directory:", current_directory)
pd.set_option("display.max_columns", None)


Current Directory: /mnt/NAS3/home/jiwon/ECTRES/python


In [2]:
# find -L /mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/results/align_dna/GRCh37 -name "*.bam" -exec realpath {} + > /mnt/NAS3/home/jiwon/ECTRES/data/bam_list_all_origin.txt

In [8]:
# 파일 경로 설정
file_path = '/mnt/NAS3/home/jiwon/ECTRES/data/bam_list_all_origin.txt'

# 파일 읽기
with open(file_path, 'r') as f:
    bam_paths = f.read().splitlines()

data = []
for path in bam_paths:
    # 경로를 'GRCh37/' 기준으로 나누어 그 바로 뒤의 폴더명을 가져옵니다.
    # 예: .../GRCh37/ECTRES-XXXX/applyBqsr/... -> ECTRES-XXXX
    try:
        barcode = path.split('GRCh37/')[1].split('/')[0]
        data.append({'aliquot_barcode': barcode, 'bam': path})
    except IndexError:
        # 경로 형식이 다를 경우를 대비한 예외 처리
        print(f"경로 형식이 맞지 않음: {path}")

# 3. 데이터프레임 생성
df = pd.DataFrame(data)

# 4. 결과 확인
print(f"총 {len(df)}개의 샘플이 로드되었습니다.")
display(df.head())

총 77개의 샘플이 로드되었습니다.


,aliquot_barcode,bam
0,ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...
1,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...
2,ECTRES-ECGI1-0001-TPX-A02-WGS-5KM079,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...
3,ECTRES-ECGI1-0001-TPX-A03-WGS-0LT586,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...
4,ECTRES-ECGI1-0001-TPX-A04-WGS-0SR571,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...


In [14]:
manifest=pd.read_csv('../manifest/ECTRES_clones_nf_dna_fastqs_20260303.csv')
manifest.head(2)

manifest["sample_id"] = manifest["sample_legacy_id"].fillna("parental")
bam_manifest = manifest[['aliquot_barcode', 'source_barcode', 'sample_barcode',
       'patient_barcode', 'sample_type', 'tumor_or_normal', 'sequence_type',
       'sample_legacy_id', 'gender','action', 'sample_id']].drop_duplicates()

print(manifest.shape, sample_mapping.shape) #(87, 19) (77, 5)
print(manifest.shape, bam_manifest.shape) #(87, 19) (77, 11)

(87, 19) (77, 5)
(87, 19) (77, 11)


In [15]:
bam_manifest.groupby('source_barcode').size()

source_barcode
ECGI1    34
EFM19    11
H2170    32
dtype: int64

In [17]:
bam_manifest= pd.merge(bam_manifest, df, on='aliquot_barcode', how='left')

In [19]:
bam_manifest['bai']=bam_manifest['bam']+'.bai'

In [20]:
bam_manifest.head(2)

,aliquot_barcode,source_barcode,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,action,sample_id,bam,bai
0,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,ECGI1,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,TP,tumor,WGS,EG_1,XY,NaN,EG_1,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...
1,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949,ECGI1,ECTRES-ECGI1-0001-TPX-A10,ECTRES-ECGI1-0001,TP,tumor,WGS,EG_10,XY,NaN,EG_10,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...


In [23]:
# bam_manifest.to_csv('../manifest/ECTRES_clones_nf_dna_bam.csv',index=False)  ## 20260309
# bam_manifest.to_csv('/mnt/NAS3/home/jiwon/ECTRES/manifest/ECTRES_clones_nf_dna_bam.csv',index=False)  ## 20260309
